# SEED BFCL OPD-only: train-128/validation-72, checkpoints 1-40

The base phase runs five epochs through checkpoint 10. Explicit compatible extensions then run deterministic epochs 6-10 through checkpoint 20 and epochs 11-20 through checkpoint 40. Every epoch has two 64-task batches, every batch produces one strict-Adam update and resumable LoRA checkpoint, and the learning rate remains constant at `1e-6` with no warmup. Held-out ordinary-prompt validation remains at epoch boundaries. Compact debugging tables, resource telemetry, health checks, and plots refresh after every extension checkpoint.

Before running the first cell, mount Google Drive with Colab's **Files > Mount Drive** control. The cell repairs a healthy `/content/drive2` fallback when present, verifies persistent write/read/delete access, and otherwise fails immediately without creating a run root.

In [ ]:
import pathlib, subprocess, uuid
CANONICAL_DRIVE = pathlib.Path('/content/drive')
FALLBACK_DRIVE = pathlib.Path('/content/drive2')
def drive_ready(root):
    return (root / 'MyDrive' / 'bfcl_qwen_experiment').is_dir()
if not drive_ready(CANONICAL_DRIVE):
    if drive_ready(FALLBACK_DRIVE):
        CANONICAL_DRIVE.mkdir(parents=True, exist_ok=True)
        subprocess.run(['sudo', 'umount', '-l', str(CANONICAL_DRIVE)], check=False)
        subprocess.run(['sudo', 'mount', '--bind', str(FALLBACK_DRIVE), str(CANONICAL_DRIVE)], check=True)
if not drive_ready(CANONICAL_DRIVE):
    raise RuntimeError(
        'Persistent Google Drive is unavailable. Use Colab Files > Mount Drive, or restore '
        'a healthy /content/drive2 mount, then rerun this cell. No run root was created.'
    )
probe = CANONICAL_DRIVE / 'MyDrive' / 'bfcl_qwen_experiment' / f'.codex_persistence_probe_{uuid.uuid4().hex}'
probe.write_text('drive-backed\n', encoding='utf-8')
assert probe.read_text(encoding='utf-8') == 'drive-backed\n'
probe.unlink()
import torch
gpu_names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
assert len(gpu_names) == 1 and 'A100' in gpu_names[0].upper(), (torch.cuda.device_count(), gpu_names)
print('SEED_BFCL_DRIVE_OK', CANONICAL_DRIVE)
print('SEED_BFCL_A100_OK', gpu_names[0])

In [ ]:
import pathlib, subprocess, sys
SEED_URL = 'https://github.com/Alexishiyu/SEED.git'
SEED_COMMIT = '852a2fb944aaf2312fd2e6840ed0dd3f68358f55'
SEED_ROOT = pathlib.Path('/content/SEED')
GORILLA_ROOT = pathlib.Path('/content/gorilla')
BFCL_ROOT = GORILLA_ROOT / 'berkeley-function-call-leaderboard'
BFCL_COMMIT = 'f7cf7359b7ac615a0b294831c5ba2bc95ee4a000'
def run(cmd, cwd=None):
    print('RUN', ' '.join(map(str, cmd)), flush=True)
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)
if not (SEED_ROOT / '.git').is_dir(): run(['git', 'clone', SEED_URL, SEED_ROOT])
run(['git', 'fetch', 'origin', SEED_COMMIT], cwd=SEED_ROOT)
seed_commit = subprocess.check_output(['git', 'rev-parse', 'FETCH_HEAD'], cwd=SEED_ROOT, text=True).strip()
assert seed_commit == SEED_COMMIT, (seed_commit, SEED_COMMIT)
run(['git', 'checkout', '--detach', seed_commit], cwd=SEED_ROOT)
if not (GORILLA_ROOT / '.git').is_dir(): run(['git', 'clone', 'https://github.com/ShishirPatil/gorilla.git', GORILLA_ROOT])
run(['git', 'fetch', 'origin', BFCL_COMMIT], cwd=GORILLA_ROOT)
run(['git', 'checkout', '--detach', BFCL_COMMIT], cwd=GORILLA_ROOT)
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip', 'setuptools', 'wheel'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm==0.11.0', 'peft==0.17.1', 'pandas', 'pyarrow', 'matplotlib'])
run([sys.executable, '-m', 'pip', 'install', '-q', 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3%2Bcu12torch2.8cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(SEED_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(BFCL_ROOT)])
run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'vllm==0.11.0'])
print('SEED_BFCL_SETUP_OK seed_sha=' + seed_commit + ' bfcl_sha=' + BFCL_COMMIT)

In [ ]:
test_files = [
    'tests/trainer/ppo/test_opd_loss.py', 'tests/trainer/ppo/test_seed_advantage.py',
    'tests/trainer/ppo/test_seed_analyzer.py', 'tests/trainer/ppo/test_episode_skill_guidance.py',
    'tests/trainer/ppo/test_seed_skill_gen_reward.py', 'tests/seed/test_june24_skill_summary.py',
    'tests/seed/test_june24_all200.py', 'tests/seed/test_june24_teacher_snapshot.py',
    'tests/trainer/ppo/test_bfcl_two_stage_optimizer.py',
    'tests/trainer/ppo/test_bfcl_checkpoint_supervisor.py',
    'tests/seed/test_bfcl_opsd_diagnostics.py',
    'tests/environments/test_bfcl_env.py', 'tests/trainer/ppo/test_opd_only_objective.py',
]
run([sys.executable, '-m', 'pytest', '-q', *test_files], cwd=SEED_ROOT)
print('SEED_BFCL_TESTS_OK')

In [ ]:
from datetime import datetime, timezone
import json
stamp = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
RUN_ROOT = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/seed_opsd_colab') / f'june24_all200_train128_val72_b64_{stamp}'
INPUTS = RUN_ROOT / 'inputs'
INPUTS.mkdir(parents=True, exist_ok=False)
SOURCE_DIRS = [
    pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/a100_skill_sd_50_20260624_055240/skills_openai'),
    pathlib.Path('/content/drive/MyDrive/bfcl_qwen_experiment/a100_skill_sd_150_50_199_20260624_063820/skills_openai'),
]
for source in SOURCE_DIRS: assert source.is_dir(), source
PAIRWISE_TASKS = pathlib.Path('/content/drive/MyDrive/bfcl_qwen_pairwise_analysis/15fRBFq4gbXgJeQ5CHO_bVjeEp0mlH9rB/pairwise_tasks.csv')
assert PAIRWISE_TASKS.is_file(), PAIRWISE_TASKS
cohort_manifest = INPUTS / 'june24_all200_cohort.json'
split_manifest = INPUTS / 'june24_train128_val72_b64_split.json'
skill_bank = INPUTS / 'june24_train128_skill_bank.json'
builder = SEED_ROOT / 'scripts/build_june24_all200_skill_bank.py'
build_cmd = [sys.executable, str(builder), '--pairwise-tasks-csv', str(PAIRWISE_TASKS), '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(split_manifest), '--output', str(skill_bank), '--split-profile', '128x72_b64', '--batch-size', '64']
for source in SOURCE_DIRS: build_cmd += ['--source-dir', str(source)]
for task_id in ['multi_turn_base_56', 'multi_turn_base_154', 'multi_turn_base_169']: build_cmd += ['--allow-repaired-task-id', task_id]
run(build_cmd, cwd=SEED_ROOT)
split_payload = json.loads(split_manifest.read_text(encoding='utf-8'))
assert split_payload['train']['classification_counts'] == {'fixed': 26, 'both_wrong': 72, 'harmed': 8, 'both_correct': 22}
assert split_payload['validation']['classification_counts'] == {'fixed': 14, 'both_wrong': 40, 'harmed': 5, 'both_correct': 13}
(RUN_ROOT / 'metadata').mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'metadata' / 'resolved_seed_commit.txt').write_text(seed_commit + '\n', encoding='utf-8')
print('SEED_BFCL_128_72_INPUTS_OK run_root=' + str(RUN_ROOT))

In [ ]:
MODEL = 'Qwen/Qwen3-4B-Instruct-2507'
RLPAPER_SHA = '690946ca3e1d2f257d7c1acdff4df4eb1feed1ec'
launcher = SEED_ROOT / 'examples/seed_trainer/run_bfcl_opsd.py'
common = [
    sys.executable, str(launcher), '--june24-skill-bank', str(skill_bank),
    '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(split_manifest),
    '--run-root', str(RUN_ROOT), '--bfcl-root', str(BFCL_ROOT), '--model', MODEL,
    '--iterations', '5', '--batch-size', '64', '--checkpoint-updates', '1',
    '--optimizer', 'adam', '--lr-schedule', 'constant', '--final-lr', '1e-6',
    '--inline-same-prompt-diagnostics', '--resume', 'auto', '--rlpaper-sha', RLPAPER_SHA,
]
run(common, cwd=SEED_ROOT)
plan = json.loads((RUN_ROOT / 'metadata/privileged_june24_plan.json').read_text(encoding='utf-8'))
assert plan['provenance']['training_rollouts'] == 640
assert plan['provenance']['validation_rollouts'] == 432
assert plan['provenance']['validation_batch_size'] == 8
assert plan['provenance']['total_updates'] == 10
assert plan['provenance']['checkpoint_schedule'] == list(range(1, 11))
assert plan['provenance']['validation_steps'] == [0, 2, 4, 6, 8, 10]
assert plan['provenance']['learning_rate'] == {'style': 'constant', 'warmup_updates': 0, 'warmup_target_lr': None, 'final_lr': 1e-6}
assert plan['provenance']['memory_profile'] == {'rollout_gpu_memory_utilization': 0.41, 'actor_activation_offload': True, 'actor_param_offload': True, 'actor_optimizer_offload': True}
print('SEED_BFCL_128_72_PREFLIGHT_OK')

In [ ]:
run(common + ['--execute', '--stop-after-update', '1'], cwd=SEED_ROOT)
checkpoint_root = RUN_ROOT / 'checkpoints' / 'privileged_june24'
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '1'
assert (RUN_ROOT / 'evidence/privileged_june24/updates/step_000001.json').is_file()
assert list((checkpoint_root / 'global_step_1/actor').glob('optim_world_size_*_rank_*.pt'))
assert (checkpoint_root / 'global_step_1/actor/lora_adapter/adapter_model.safetensors').is_file()
print('SEED_BFCL_BATCH64_DRIVE_SMOKE_OK checkpoint=1')

In [ ]:
run(common + ['--execute'], cwd=SEED_ROOT)
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '10'
print('SEED_BFCL_128_72_FIVE_EPOCH_TRAIN_OK')

In [ ]:
extension_inputs = INPUTS / 'extension_to20'
extension_inputs.mkdir(parents=True, exist_ok=True)
extension_split_manifest = extension_inputs / 'june24_train128_val72_b64_split.json'
extension_skill_bank = extension_inputs / 'june24_train128_skill_bank.json'
extension_build_cmd = [sys.executable, str(builder), '--pairwise-tasks-csv', str(PAIRWISE_TASKS), '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(extension_split_manifest), '--output', str(extension_skill_bank), '--split-profile', '128x72_b64', '--batch-size', '64', '--iterations', '10']
for source in SOURCE_DIRS: extension_build_cmd += ['--source-dir', str(source)]
for task_id in ['multi_turn_base_56', 'multi_turn_base_154', 'multi_turn_base_169']: extension_build_cmd += ['--allow-repaired-task-id', task_id]
run(extension_build_cmd, cwd=SEED_ROOT)
base_split = json.loads(split_manifest.read_text(encoding='utf-8'))
extension_split = json.loads(extension_split_manifest.read_text(encoding='utf-8'))
assert extension_split['train'] == base_split['train']
assert extension_split['validation'] == base_split['validation']
assert extension_split['training_schedule']['iteration_schedules'][:5] == base_split['training_schedule']['iteration_schedules']
assert extension_split['training_schedule']['total_updates'] == 20
assert extension_split['training_schedule']['total_rollouts'] == 1280
print('SEED_BFCL_EXTENSION_INPUTS_OK checkpoint=10 target=20')

In [ ]:
extension_common = [
    sys.executable, str(launcher), '--june24-skill-bank', str(extension_skill_bank),
    '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(extension_split_manifest),
    '--run-root', str(RUN_ROOT), '--bfcl-root', str(BFCL_ROOT), '--model', MODEL,
    '--iterations', '10', '--extend-from-update', '10', '--batch-size', '64', '--checkpoint-updates', '1',
    '--optimizer', 'adam', '--lr-schedule', 'constant', '--final-lr', '1e-6',
    '--inline-same-prompt-diagnostics', '--resume', 'auto', '--rlpaper-sha', RLPAPER_SHA,
]
run(extension_common, cwd=SEED_ROOT)
extension_plan = json.loads((RUN_ROOT / 'metadata/privileged_june24_plan.json').read_text(encoding='utf-8'))
assert extension_plan['provenance']['extension_parent']['extension_from_update'] == 10
assert extension_plan['provenance']['total_updates'] == 20
assert extension_plan['provenance']['training_rollouts'] == 1280
assert extension_plan['provenance']['validation_rollouts'] == 792
assert extension_plan['provenance']['checkpoint_schedule'] == list(range(1, 21))
assert extension_plan['provenance']['validation_steps'] == list(range(0, 21, 2))
print('SEED_BFCL_EXTENSION_PREFLIGHT_OK checkpoint=10 target=20')

In [ ]:
run(extension_common + ['--execute', '--stop-after-update', '11'], cwd=SEED_ROOT)
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '11'
step11 = json.loads((RUN_ROOT / 'evidence/privileged_june24/updates/step_000011.json').read_text(encoding='utf-8'))
assert step11['iteration'] == 6 and step11['batch_in_iteration'] == 1
assert len(step11['task_token_metrics']) == 64
debug11 = json.loads((RUN_ROOT / 'evidence/privileged_june24/diagnostics/diagnostic_summary.json').read_text(encoding='utf-8'))
assert debug11['status'] == 'healthy' and debug11['latest_checkpoint'] == 11
print('SEED_BFCL_EXTENSION_SMOKE_OK checkpoint=11')

In [ ]:
run(extension_common + ['--execute'], cwd=SEED_ROOT)
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '20'
assert (RUN_ROOT / 'evidence/privileged_june24/validation/global_step_000020.json').is_file()
debug20 = json.loads((RUN_ROOT / 'evidence/privileged_june24/diagnostics/diagnostic_summary.json').read_text(encoding='utf-8'))
assert debug20['status'] == 'healthy' and debug20['latest_checkpoint'] == 20
print('SEED_BFCL_128_72_TEN_EPOCH_TRAIN_OK checkpoint=20')

In [ ]:
extension40_inputs = INPUTS / 'extension_to40'
extension40_inputs.mkdir(parents=True, exist_ok=True)
extension40_split_manifest = extension40_inputs / 'june24_train128_val72_b64_split.json'
extension40_skill_bank = extension40_inputs / 'june24_train128_skill_bank.json'
extension40_build_cmd = [sys.executable, str(builder), '--pairwise-tasks-csv', str(PAIRWISE_TASKS), '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(extension40_split_manifest), '--output', str(extension40_skill_bank), '--split-profile', '128x72_b64', '--batch-size', '64', '--iterations', '20']
for source in SOURCE_DIRS: extension40_build_cmd += ['--source-dir', str(source)]
for task_id in ['multi_turn_base_56', 'multi_turn_base_154', 'multi_turn_base_169']: extension40_build_cmd += ['--allow-repaired-task-id', task_id]
run(extension40_build_cmd, cwd=SEED_ROOT)
extension40_split = json.loads(extension40_split_manifest.read_text(encoding='utf-8'))
assert extension40_split['train'] == extension_split['train']
assert extension40_split['validation'] == extension_split['validation']
assert extension40_split['training_schedule']['iteration_schedules'][:10] == extension_split['training_schedule']['iteration_schedules']
assert extension40_split['training_schedule']['total_updates'] == 40
assert extension40_split['training_schedule']['total_rollouts'] == 2560
print('SEED_BFCL_EXTENSION40_INPUTS_OK checkpoint=20 target=40')

In [ ]:
extension40_common = [
    sys.executable, str(launcher), '--june24-skill-bank', str(extension40_skill_bank),
    '--cohort-manifest', str(cohort_manifest), '--split-manifest', str(extension40_split_manifest),
    '--run-root', str(RUN_ROOT), '--bfcl-root', str(BFCL_ROOT), '--model', MODEL,
    '--iterations', '20', '--extend-from-update', '20', '--batch-size', '64', '--checkpoint-updates', '1',
    '--optimizer', 'adam', '--lr-schedule', 'constant', '--final-lr', '1e-6',
    '--inline-same-prompt-diagnostics', '--resume', 'auto', '--rlpaper-sha', RLPAPER_SHA,
]
run(extension40_common, cwd=SEED_ROOT)
extension40_plan = json.loads((RUN_ROOT / 'metadata/privileged_june24_plan.json').read_text(encoding='utf-8'))
assert extension40_plan['provenance']['extension_parent']['extension_from_update'] == 20
assert extension40_plan['provenance']['extension_parent']['target_update'] == 40
assert extension40_plan['provenance']['total_updates'] == 40
assert extension40_plan['provenance']['training_rollouts'] == 2560
assert extension40_plan['provenance']['validation_rollouts'] == 1512
assert extension40_plan['provenance']['checkpoint_schedule'] == list(range(1, 41))
assert extension40_plan['provenance']['validation_steps'] == list(range(0, 41, 2))
print('SEED_BFCL_EXTENSION40_PREFLIGHT_OK checkpoint=20 target=40')

In [ ]:
run(extension40_common + ['--execute', '--stop-after-update', '21'], cwd=SEED_ROOT)
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '21'
step21 = json.loads((RUN_ROOT / 'evidence/privileged_june24/updates/step_000021.json').read_text(encoding='utf-8'))
assert step21['iteration'] == 11 and step21['batch_in_iteration'] == 1
assert len(step21['task_token_metrics']) == 64
debug21 = json.loads((RUN_ROOT / 'evidence/privileged_june24/diagnostics/diagnostic_summary.json').read_text(encoding='utf-8'))
assert debug21['status'] == 'healthy' and debug21['latest_checkpoint'] == 21
print('SEED_BFCL_EXTENSION40_SMOKE_OK checkpoint=21')

In [ ]:
run(extension40_common + ['--execute'], cwd=SEED_ROOT)
assert (checkpoint_root / 'latest_checkpointed_iteration.txt').read_text().strip() == '40'
assert (RUN_ROOT / 'evidence/privileged_june24/validation/global_step_000040.json').is_file()
debug40 = json.loads((RUN_ROOT / 'evidence/privileged_june24/diagnostics/diagnostic_summary.json').read_text(encoding='utf-8'))
assert debug40['status'] == 'healthy' and debug40['latest_checkpoint'] == 40
print('SEED_BFCL_128_72_TWENTY_EPOCH_TRAIN_OK checkpoint=40')

In [ ]:
export_root = RUN_ROOT / 'exports' / 'privileged_june24_merged'
export_evidence = RUN_ROOT / 'evidence' / 'privileged_june24' / 'checkpoint_export.json'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/merge_bfcl_opsd_lora.py'), '--checkpoint-root', str(checkpoint_root), '--base-model', MODEL, '--output', str(export_root), '--evidence', str(export_evidence), '--validate-vllm'], cwd=SEED_ROOT)
print('SEED_BFCL_FINAL_MERGE_RELOAD_OK')

In [ ]:
final_report = RUN_ROOT / 'evidence' / 'seed_bfcl_checkpoint40_complete.json'
run([sys.executable, str(SEED_ROOT / 'examples/seed_trainer/collect_bfcl_opsd_five_pass_evidence.py'), '--run-root', str(RUN_ROOT), '--export-evidence', str(export_evidence), '--output', str(final_report)], cwd=SEED_ROOT)
report = json.loads(final_report.read_text(encoding='utf-8'))
assert report['status'] == 'complete'
print('SEED_BFCL_128_72_CHECKPOINT40_COMPLETE', final_report)
print(json.dumps(report['validation_curve'], indent=2))
from IPython.display import display, Image
diagnostic_root = RUN_ROOT / 'evidence/privileged_june24/diagnostics'
for plot_name in ['training_diagnostics.png', 'validation_diagnostics.png', 'validation_task_transitions.png', 'runtime_resources.png']:
    plot_path = diagnostic_root / plot_name
    if plot_path.is_file(): display(Image(filename=str(plot_path)))